Celda 1: Importación de librerías y carga de datos de augmentation

In [2]:
import pandas as pd
import numpy as np
import os
import time
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

BASE_DIR = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed"
PATH_MENSUAL = os.path.join(BASE_DIR, 'dataset', 'dataset_mensual.xlsx')
PATH_BOOTSTRAP = os.path.join(BASE_DIR, 'augmented', 'dataset_bootstrap.xlsx')
PATH_SMOTE = os.path.join(BASE_DIR, 'augmented', 'dataset_smote.xlsx')
PATH_OUTPUT_DIR = os.path.join(BASE_DIR, 'resultados')
PATH_GRAFICAS = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\notebooks\prediction\graficas"

os.makedirs(PATH_OUTPUT_DIR, exist_ok=True)
os.makedirs(PATH_GRAFICAS, exist_ok=True)

df_mensual_base = pd.read_excel(PATH_MENSUAL)
df_bootstrap_raw = pd.read_excel(PATH_BOOTSTRAP)
df_smote_raw = pd.read_excel(PATH_SMOTE)

print(f"Dataset mensual base: {len(df_mensual_base)} meses")
print(f"Dataset bootstrap (registros): {len(df_bootstrap_raw)} registros")
print(f"Dataset SMOTE (registros): {len(df_smote_raw)} registros")
print(f"Columnas bootstrap: {list(df_bootstrap_raw.columns[:10])}...")

Importing plotly failed. Interactive plots will not work.


Dataset mensual base: 46 meses
Dataset bootstrap (registros): 2040 registros
Dataset SMOTE (registros): 836 registros
Columnas bootstrap: ['Fecha', 'Forma de pago', 'Ataud_Modelo', 'Ataud_Color', 'Capilla', 'Carroza', 'Carroza flores', 'Cargadores', 'Auto', 'Microbus']...


Celda 2: Agregación de bootstrap a formato mensual

In [3]:
def agregar_a_mensual(df_registro, nombre_fuente):
    df = df_registro.copy()
    if 'Fecha' not in df.columns:
        print(f"  [SKIP] {nombre_fuente}: no tiene columna 'Fecha'")
        return None

    df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce')
    df = df.dropna(subset=['Fecha'])
    df['Periodo'] = df['Fecha'].dt.to_period('M')

    cols_binarias = [c for c in ['Carroza', 'Carroza flores', 'Auto', 'Microbus'] if c in df.columns]
    agg_dict = {
        'Fecha': 'count',
    }
    if 'Monto' in df.columns:
        agg_dict['Monto'] = ['sum', 'mean', 'median']
    for col in cols_binarias:
        agg_dict[col] = 'sum'
    if 'Cargadores' in df.columns:
        agg_dict['Cargadores'] = 'sum'

    df_mensual = df.groupby('Periodo').agg(agg_dict)

    if isinstance(df_mensual.columns, pd.MultiIndex):
        new_cols = []
        for col in df_mensual.columns:
            if col[1] == '':
                new_cols.append(col[0])
            else:
                new_cols.append(f"{col[0]}_{col[1]}")
        df_mensual.columns = new_cols

    rename_map = {'Fecha_count': 'servicios_totales'}
    if 'Monto_sum' in df_mensual.columns:
        rename_map['Monto_sum'] = 'monto_total'
    if 'Monto_mean' in df_mensual.columns:
        rename_map['Monto_mean'] = 'monto_promedio'
    if 'Monto_median' in df_mensual.columns:
        rename_map['Monto_median'] = 'monto_mediana'
    if 'Carroza_sum' in df_mensual.columns:
        rename_map['Carroza_sum'] = 'carroza_count'
    if 'Carroza flores_sum' in df_mensual.columns:
        rename_map['Carroza flores_sum'] = 'carroza_flores_count'
    if 'Auto_sum' in df_mensual.columns:
        rename_map['Auto_sum'] = 'auto_count'
    if 'Microbus_sum' in df_mensual.columns:
        rename_map['Microbus_sum'] = 'microbus_count'
    if 'Cargadores_sum' in df_mensual.columns:
        rename_map['Cargadores_sum'] = 'cargadores_total'
    df_mensual = df_mensual.rename(columns=rename_map)

    for col_pivot in ['Ataud_Modelo', 'Ataud_Color', 'Capilla']:
        if col_pivot in df.columns:
            pivot = df.pivot_table(index='Periodo', columns=col_pivot, values='Monto', aggfunc='count', fill_value=0)
            pivot.columns = [f'{col_pivot}_{c}' for c in pivot.columns]
            df_mensual = df_mensual.join(pivot, how='left')

    df_mensual = df_mensual.fillna(0)

    all_months = pd.period_range(start=df['Periodo'].min(), end=df['Periodo'].max(), freq='M')
    df_mensual = df_mensual.reindex(all_months, fill_value=0)
    df_mensual.index.name = 'Periodo'
    df_mensual = df_mensual.reset_index()

    print(f"  {nombre_fuente}: {len(df_mensual)} meses, columnas: {list(df_mensual.columns[:8])}...")
    return df_mensual

print("Agregación a formato mensual:")
df_mensual_bootstrap = agregar_a_mensual(df_bootstrap_raw, "Bootstrap")
df_mensual_smote = agregar_a_mensual(df_smote_raw, "SMOTE")

Agregación a formato mensual:
  Bootstrap: 46 meses, columnas: ['Periodo', 'servicios_totales', 'monto_total', 'monto_promedio', 'monto_mediana', 'carroza_count', 'carroza_flores_count', 'auto_count']...
  [SKIP] SMOTE: no tiene columna 'Fecha'


Celda 3: Verificación de datasets mensuales generados

In [4]:
print("=== Datasets mensuales disponibles ===")
print()

datasets_mensuales = {
    'base': df_mensual_base,
}
if df_mensual_bootstrap is not None:
    datasets_mensuales['bootstrap'] = df_mensual_bootstrap
if df_mensual_smote is not None:
    datasets_mensuales['smote'] = df_mensual_smote

for nombre, ds in datasets_mensuales.items():
    if 'Periodo' in ds.columns:
        ds_check = ds.set_index('Periodo')
    else:
        ds_check = ds
    print(f"{nombre.upper()}:")
    print(f"  Meses: {len(ds_check)}")
    if 'servicios_totales' in ds_check.columns:
        print(f"  Servicios totales - media: {ds_check['servicios_totales'].mean():.1f}, std: {ds_check['servicios_totales'].std():.1f}")
    if 'monto_total' in ds_check.columns:
        print(f"  Monto total - media: {ds_check['monto_total'].mean():.0f}, std: {ds_check['monto_total'].std():.0f}")
    print()

=== Datasets mensuales disponibles ===

BASE:
  Meses: 46
  Servicios totales - media: 7.4, std: 6.0
  Monto total - media: 81203, std: 189023

BOOTSTRAP:
  Meses: 46
  Servicios totales - media: 44.3, std: 37.5
  Monto total - media: 519433, std: 1222002



Celda 4: Generación de Sliding Window desde bootstrap

In [5]:
def crear_sliding_windows(df_mensual, window_size=3, targets=None):
    if targets is None:
        targets = ['servicios_totales', 'monto_total']

    df_win = df_mensual.copy()
    if 'Periodo' in df_win.columns:
        df_win = df_win.set_index('Periodo')

    windows = []
    for i in range(window_size, len(df_win)):
        window = {}
        for target in targets:
            if target in df_win.columns:
                for lag in range(1, window_size + 1):
                    window[f'{target}_lag_{lag}'] = df_win[target].iloc[i - lag]
                window[f'{target}_target'] = df_win[target].iloc[i]
        window['mes_target'] = str(df_win.index[i])
        windows.append(window)

    return pd.DataFrame(windows)

targets_disponibles = ['servicios_totales', 'monto_total']

df_windows_bootstrap = None
if df_mensual_bootstrap is not None:
    df_windows_bootstrap = crear_sliding_windows(df_mensual_bootstrap, window_size=3, targets=targets_disponibles)
    print(f"Sliding windows desde bootstrap: {len(df_windows_bootstrap)} ventanas")
    print(f"Columnas: {list(df_windows_bootstrap.columns)}")
else:
    print("No se generaron sliding windows (bootstrap no disponible)")

Sliding windows desde bootstrap: 43 ventanas
Columnas: ['servicios_totales_lag_1', 'servicios_totales_lag_2', 'servicios_totales_lag_3', 'servicios_totales_target', 'monto_total_lag_1', 'monto_total_lag_2', 'monto_total_lag_3', 'monto_total_target', 'mes_target']


Celda 5: Split temporal para las 3 fuentes de datos

In [6]:
TARGETS = ['servicios_totales', 'monto_total']
test_size = 12

def preparar_fuente(df_mensual, nombre):
    if df_mensual is None:
        return None, None
    ds = df_mensual.copy()
    if 'Periodo' in ds.columns:
        ds['Periodo'] = pd.to_datetime(ds['Periodo'].astype(str))
        ds = ds.set_index('Periodo')
    if len(ds) <= test_size:
        print(f"  [SKIP] {nombre}: solo {len(ds)} meses, insuficiente para split")
        return None, None
    train = ds.iloc[:-test_size]
    test = ds.iloc[-test_size:]
    print(f"{nombre}: Train {len(train)} meses ({train.index[0].date()} a {train.index[-1].date()})")
    print(f"         Test  {len(test)} meses ({test.index[0].date()} a {test.index[-1].date()})")
    return train, test

print("=== Split Temporal ===")
splits = {}
for nombre, ds in datasets_mensuales.items():
    train, test = preparar_fuente(ds, nombre)
    if train is not None:
        splits[nombre] = {'train': train, 'test': test}
    print()

=== Split Temporal ===
base: Train 34 meses (2022-05-01 a 2025-02-01)
         Test  12 meses (2025-03-01 a 2026-02-01)

bootstrap: Train 34 meses (2022-05-01 a 2025-02-01)
         Test  12 meses (2025-03-01 a 2026-02-01)



Celda 6: SARIMA con las 3 fuentes de datos

In [7]:
def entrenar_evaluar_sarima(y_train, y_test):
    start = time.time()
    model = SARIMAX(y_train, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0),
                    enforce_stationarity=False, enforce_invertibility=False)
    results = model.fit(disp=False)
    pred = results.forecast(steps=len(y_test))
    tiempo = time.time() - start
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred) if len(y_test) > 1 else float('nan')
    mape = np.mean(np.abs((y_test.values - pred.values) / np.where(y_test.values == 0, 1, y_test.values))) * 100
    return {
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

resultados_sarima = []
print("=== SARIMA ===")
for fuente, split in splits.items():
    for target in TARGETS:
        y_train = split['train'][target]
        y_test = split['test'][target]
        res = entrenar_evaluar_sarima(y_train, y_test)
        res['modelo'] = 'SARIMA'
        res['target'] = target
        res['fuente'] = fuente
        resultados_sarima.append(res)
        print(f"  SARIMA [{fuente}] {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo']:.2f}s")

=== SARIMA ===
  SARIMA [base] servicios_totales: MAE=6.53, R2=-1.1950, Tiempo=0.01s
  SARIMA [base] monto_total: MAE=45656.64, R2=-0.1029, Tiempo=0.02s
  SARIMA [bootstrap] servicios_totales: MAE=45.91, R2=-1.7998, Tiempo=0.02s
  SARIMA [bootstrap] monto_total: MAE=285053.42, R2=-0.2669, Tiempo=0.02s


Celda 7: Prophet con las 3 fuentes de datos

In [8]:
def entrenar_evaluar_prophet(y_train, y_test):
    start = time.time()
    df_p = pd.DataFrame({'ds': y_train.index, 'y': y_train.values})
    model = Prophet(yearly_seasonality=False, weekly_seasonality=False,
                    daily_seasonality=False, changepoint_prior_scale=0.5)
    model.fit(df_p)
    future = pd.DataFrame({'ds': y_test.index})
    forecast = model.predict(future)
    pred = forecast['yhat'].values[:len(y_test)]
    tiempo = time.time() - start
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred) if len(y_test) > 1 else float('nan')
    mape = np.mean(np.abs((y_test.values - pred) / np.where(y_test.values == 0, 1, y_test.values))) * 100
    return {
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

resultados_prophet = []
print("=== Prophet ===")
for fuente, split in splits.items():
    for target in TARGETS:
        y_train = split['train'][target]
        y_test = split['test'][target]
        res = entrenar_evaluar_prophet(y_train, y_test)
        res['modelo'] = 'Prophet'
        res['target'] = target
        res['fuente'] = fuente
        resultados_prophet.append(res)
        print(f"  Prophet [{fuente}] {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo']:.2f}s")

=== Prophet ===


01:20:33 - cmdstanpy - INFO - Chain [1] start processing
01:20:33 - cmdstanpy - INFO - Chain [1] done processing
01:20:33 - cmdstanpy - INFO - Chain [1] start processing


  Prophet [base] servicios_totales: MAE=4.31, R2=0.0064, Tiempo=0.35s


01:20:33 - cmdstanpy - INFO - Chain [1] done processing
01:20:33 - cmdstanpy - INFO - Chain [1] start processing


  Prophet [base] monto_total: MAE=90965.17, R2=-1.1035, Tiempo=0.39s


01:20:33 - cmdstanpy - INFO - Chain [1] done processing
01:20:34 - cmdstanpy - INFO - Chain [1] start processing


  Prophet [bootstrap] servicios_totales: MAE=28.30, R2=-0.0008, Tiempo=0.28s


01:20:34 - cmdstanpy - INFO - Chain [1] done processing


  Prophet [bootstrap] monto_total: MAE=905430.16, R2=-3.8378, Tiempo=0.48s


Celda 8: LSTM con las 3 fuentes de datos

In [9]:
def entrenar_evaluar_lstm(y_train, y_test, window_size=3):
    start = time.time()

    scaler = MinMaxScaler()
    y_all = np.concatenate([y_train.values, y_test.values])
    y_scaled_all = scaler.fit_transform(y_all.reshape(-1, 1))
    y_scaled = y_scaled_all[:len(y_train)]

    if len(y_scaled) < window_size:
        return None

    X_train_lstm, y_train_lstm = [], []
    for i in range(window_size, len(y_scaled)):
        X_train_lstm.append(y_scaled[i - window_size:i, 0])
        y_train_lstm.append(y_scaled[i, 0])
    X_train_lstm = np.array(X_train_lstm).reshape(-1, window_size, 1)
    y_train_lstm = np.array(y_train_lstm)

    model = Sequential([
        LSTM(32, input_shape=(window_size, 1)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_train_lstm, y_train_lstm, epochs=50, batch_size=4, verbose=0)

    predicciones = []
    sequence = y_scaled[-window_size:].flatten().tolist()
    for _ in range(len(y_test)):
        x_input = np.array(sequence[-window_size:]).reshape(1, window_size, 1)
        pred_scaled = model.predict(x_input, verbose=0)[0, 0]
        predicciones.append(pred_scaled)
        sequence.append(pred_scaled)

    pred_values = scaler.inverse_transform(
        np.array(predicciones).reshape(-1, 1)
    ).flatten()

    tiempo = time.time() - start
    y_true = y_test.values
    mae = mean_absolute_error(y_true, pred_values)
    rmse = np.sqrt(mean_squared_error(y_true, pred_values))
    r2 = r2_score(y_true, pred_values) if len(y_true) > 1 else float('nan')
    mape = np.mean(np.abs((y_true - pred_values) / np.where(y_true == 0, 1, y_true))) * 100

    return {
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo': tiempo,
        'predicciones': pred_values.tolist(), 'real': y_true.tolist()
    }

resultados_lstm = []
print("=== LSTM ===")
for fuente, split in splits.items():
    for target in TARGETS:
        y_train = split['train'][target]
        y_test = split['test'][target]
        res = entrenar_evaluar_lstm(y_train, y_test)
        if res is not None:
            res['modelo'] = 'LSTM'
            res['target'] = target
            res['fuente'] = fuente
            resultados_lstm.append(res)
            print(f"  LSTM [{fuente}] {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo']:.2f}s")
        else:
            print(f"  LSTM [{fuente}] {target}: SKIP (datos insuficientes)")

=== LSTM ===
  LSTM [base] servicios_totales: MAE=5.50, R2=-0.4956, Tiempo=5.20s
  LSTM [base] monto_total: MAE=80668.31, R2=-0.0823, Tiempo=4.99s
  LSTM [bootstrap] servicios_totales: MAE=31.16, R2=-0.3115, Tiempo=4.88s
  LSTM [bootstrap] monto_total: MAE=400316.89, R2=-0.0237, Tiempo=5.00s


Celda 9: ETS con las 3 fuentes de datos

In [10]:
def entrenar_evaluar_ets(y_train, y_test):
    start = time.time()
    try:
        model = ExponentialSmoothing(y_train, trend='add', seasonal=None,
                                     initialization_method='estimated')
        results = model.fit(optimized=True)
        pred = results.forecast(steps=len(y_test))
    except Exception:
        model = ExponentialSmoothing(y_train, trend='add', seasonal=None,
                                     initialization_method='heuristic')
        results = model.fit(optimized=True)
        pred = results.forecast(steps=len(y_test))
    tiempo = time.time() - start
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred) if len(y_test) > 1 else float('nan')
    mape = np.mean(np.abs((y_test.values - pred.values) / np.where(y_test.values == 0, 1, y_test.values))) * 100
    return {
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

resultados_ets = []
print("=== ETS ===")
for fuente, split in splits.items():
    for target in TARGETS:
        y_train = split['train'][target]
        y_test = split['test'][target]
        res = entrenar_evaluar_ets(y_train, y_test)
        res['modelo'] = 'ETS'
        res['target'] = target
        res['fuente'] = fuente
        resultados_ets.append(res)
        print(f"  ETS [{fuente}] {target}: MAE={res['MAE']:.2f}, R2={res['R2']:.4f}, Tiempo={res['tiempo']:.2f}s")

=== ETS ===
  ETS [base] servicios_totales: MAE=5.09, R2=-0.3561, Tiempo=0.02s
  ETS [base] monto_total: MAE=150505.63, R2=-1.6464, Tiempo=0.02s
  ETS [bootstrap] servicios_totales: MAE=36.55, R2=-0.7764, Tiempo=0.02s
  ETS [bootstrap] monto_total: MAE=270232.43, R2=-0.0329, Tiempo=0.02s


Celda 10: XGBoost y LightGBM con Sliding Window del bootstrap

In [11]:
def entrenar_evaluar_xgboost(df_windows, target):
    start = time.time()
    lag_cols = [f'{target}_lag_1', f'{target}_lag_2', f'{target}_lag_3']
    target_col = f'{target}_target'

    if not all(c in df_windows.columns for c in lag_cols + [target_col]):
        return None

    X = df_windows[lag_cols]
    y = df_windows[target_col]

    if len(X) < 2:
        return None

    split_idx = max(1, int(len(X) * 0.8))
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                         random_state=42, verbosity=0)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    tiempo = time.time() - start

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred) if len(y_test) > 1 else 0.0
    mape = np.mean(np.abs((y_test.values - pred) / np.where(y_test.values == 0, 1, y_test.values))) * 100

    return {
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

def entrenar_evaluar_lgbm(df_windows, target):
    start = time.time()
    lag_cols = [f'{target}_lag_1', f'{target}_lag_2', f'{target}_lag_3']
    target_col = f'{target}_target'

    if not all(c in df_windows.columns for c in lag_cols + [target_col]):
        return None

    X = df_windows[lag_cols]
    y = df_windows[target_col]

    if len(X) < 2:
        return None

    split_idx = max(1, int(len(X) * 0.8))
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = LGBMRegressor(num_leaves=15, n_estimators=100, learning_rate=0.05,
                          random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    tiempo = time.time() - start

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred) if len(y_test) > 1 else 0.0
    mape = np.mean(np.abs((y_test.values - pred) / np.where(y_test.values == 0, 1, y_test.values))) * 100

    return {
        'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape,
        'tiempo': tiempo,
        'predicciones': pred.tolist(), 'real': y_test.tolist()
    }

resultados_ml = []
print("=== XGBoost + LightGBM (Sliding Window Bootstrap) ===")
if df_windows_bootstrap is not None:
    for target in TARGETS:
        res_xgb = entrenar_evaluar_xgboost(df_windows_bootstrap, target)
        if res_xgb is not None:
            res_xgb['modelo'] = 'XGBoost'
            res_xgb['target'] = target
            res_xgb['fuente'] = 'bootstrap_window'
            resultados_ml.append(res_xgb)
            print(f"  XGBoost [bootstrap] {target}: MAE={res_xgb['MAE']:.2f}, R2={res_xgb['R2']:.4f}, Tiempo={res_xgb['tiempo']:.2f}s")

        res_lgbm = entrenar_evaluar_lgbm(df_windows_bootstrap, target)
        if res_lgbm is not None:
            res_lgbm['modelo'] = 'LightGBM'
            res_lgbm['target'] = target
            res_lgbm['fuente'] = 'bootstrap_window'
            resultados_ml.append(res_lgbm)
            print(f"  LightGBM [bootstrap] {target}: MAE={res_lgbm['MAE']:.2f}, R2={res_lgbm['R2']:.4f}, Tiempo={res_lgbm['tiempo']:.2f}s")
else:
    print("  No hay sliding windows disponibles del bootstrap")

=== XGBoost + LightGBM (Sliding Window Bootstrap) ===
  XGBoost [bootstrap] servicios_totales: MAE=32.12, R2=-0.2410, Tiempo=1.13s
  LightGBM [bootstrap] servicios_totales: MAE=31.92, R2=-0.5918, Tiempo=1.42s
  XGBoost [bootstrap] monto_total: MAE=317631.67, R2=-0.2032, Tiempo=0.04s
  LightGBM [bootstrap] monto_total: MAE=512494.33, R2=-0.1347, Tiempo=0.02s


Celda 11: Tabla comparativa y gráficos

In [12]:
todos_resultados = (
    resultados_sarima + resultados_prophet +
    resultados_lstm + resultados_ets + resultados_ml
)

df_comparativa = pd.DataFrame(todos_resultados)
cols_export = ['modelo', 'fuente', 'target', 'MAE', 'RMSE', 'R2', 'MAPE', 'tiempo']
cols_existentes = [c for c in cols_export if c in df_comparativa.columns]
df_comparativa = df_comparativa[cols_existentes]

print("=" * 80)
print("TABLA COMPARATIVA - Modelos con Data Augmentation")
print("=" * 80)
print()
print(df_comparativa.to_string(index=False))
print()

print("\n=== MEJORES MODELOS POR FUENTE ===")
for fuente in df_comparativa['fuente'].unique():
    subset = df_comparativa[df_comparativa['fuente'] == fuente]
    for target in TARGETS:
        sub_target = subset[subset['target'] == target]
        if len(sub_target) > 0:
            mejor = sub_target.loc[sub_target['R2'].idxmax()]
            print(f"  [{fuente}] {target}: {mejor['modelo']} (R2={mejor['R2']:.4f}, MAE={mejor['MAE']:.2f})")

TABLA COMPARATIVA - Modelos con Data Augmentation

  modelo           fuente            target           MAE         RMSE        R2        MAPE   tiempo
  SARIMA             base servicios_totales      6.533428 7.966016e+00 -1.195020   64.170234 0.014829
  SARIMA             base       monto_total  45656.640954 9.983176e+04 -0.102891  145.281012 0.020279
  SARIMA        bootstrap servicios_totales     45.907617 5.666482e+01 -1.799798   64.512250 0.021172
  SARIMA        bootstrap       monto_total 285053.416663 5.743558e+05 -0.266868   92.048077 0.017679
 Prophet             base servicios_totales      4.314616 5.359541e+00  0.006401   94.009825 0.346402
 Prophet             base       monto_total  90965.169217 1.378696e+05 -1.103451  398.611508 0.386860
 Prophet        bootstrap servicios_totales     28.301076 3.387842e+01 -0.000797  100.893664 0.279825
 Prophet        bootstrap       monto_total 905430.159336 1.122376e+06 -3.837779  853.808243 0.479267
    LSTM             base servi

Celda 11.1: Gráficos comparativos por fuente de datos

In [13]:
fuentes = df_comparativa['fuente'].unique()
modelos = df_comparativa['modelo'].unique()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, target in enumerate(TARGETS):
    for j, metric in enumerate(['MAE', 'RMSE', 'R2']):
        ax = axes[idx * 3 + j]
        subset = df_comparativa[df_comparativa['target'] == target]
        x_pos = np.arange(len(modelos))
        width = 0.25

        for i, fuente in enumerate(fuentes):
            data_fuente = subset[subset['fuente'] == fuente]
            vals = []
            for m in modelos:
                row = data_fuente[data_fuente['modelo'] == m]
                vals.append(row[metric].values[0] if len(row) > 0 else 0)
            ax.bar(x_pos + i * width, vals, width, label=fuente, alpha=0.8)

        ax.set_title(f"{target} - {metric}", fontsize=10)
        ax.set_xticks(x_pos + width)
        ax.set_xticklabels(modelos, rotation=45, fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

plt.suptitle("Comparación de Modelos por Fuente de Datos (Augmentation)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PATH_GRAFICAS, 'augmented_comparativa_mae_rmse.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Gráfico guardado: augmented_comparativa_mae_rmse.png")

Gráfico guardado: augmented_comparativa_mae_rmse.png


Celda 11.2: Gráficos de predicciones por modelo y fuente

In [14]:
modelos_nombres = {
    'SARIMA': 'sarima', 'Prophet': 'prophet', 'XGBoost': 'xgboost',
    'LightGBM': 'lgbm', 'LSTM': 'lstm', 'ETS': 'ets'
}

for target in TARGETS:
    target_results = [r for r in todos_resultados if r['target'] == target]
    n_modelos = len(set(r['modelo'] for r in target_results))
    n_fuentes = len(set(r['fuente'] for r in target_results))

    fig, axes = plt.subplots(n_fuentes, n_modelos, figsize=(5 * n_modelos, 4 * n_fuentes))
    if n_fuentes == 1:
        axes = axes.reshape(1, -1)
    if n_modelos == 1:
        axes = axes.reshape(-1, 1)

    for i, fuente in enumerate(sorted(set(r['fuente'] for r in target_results))):
        for j, modelo in enumerate(sorted(set(r['modelo'] for r in target_results))):
            ax = axes[i, j]
            res = [r for r in target_results if r['fuente'] == fuente and r['modelo'] == modelo]
            if len(res) > 0:
                r = res[0]
                x_real = range(len(r['real']))
                x_pred = range(len(r['predicciones']))
                ax.plot(x_real, r['real'], 'b-o', label='Real', markersize=4)
                ax.plot(x_pred, r['predicciones'], 'r--x', label='Predicho', markersize=4)
                ax.set_title(f"{modelo}\nMAE={r['MAE']:.1f}, R2={r['R2']:.3f}", fontsize=8)
                ax.legend(fontsize=7)
            ax.set_xlabel(f"Fuente: {fuente}", fontsize=7)
            ax.grid(True, alpha=0.3)

    plt.suptitle(f"Predicciones - {target}", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(PATH_GRAFICAS, f'augmented_predicciones_{target}.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Gráfico guardado: augmented_predicciones_{target}.png")

Gráfico guardado: augmented_predicciones_servicios_totales.png
Gráfico guardado: augmented_predicciones_monto_total.png


Celda 12: Exportación de resultados

In [15]:
df_comparativa.to_excel(os.path.join(PATH_OUTPUT_DIR, 'augmented_comparativa_modelos.xlsx'), index=False)
print(f"  Guardado: augmented_comparativa_modelos.xlsx")

predicciones_rows = []
for res in todos_resultados:
    for i, (real, pred) in enumerate(zip(res['real'], res['predicciones'])):
        predicciones_rows.append({
            'modelo': res['modelo'],
            'fuente': res['fuente'],
            'target': res['target'],
            'punto': i,
            'real': real,
            'predicho': pred,
            'error_absoluto': abs(real - pred)
        })
df_predicciones = pd.DataFrame(predicciones_rows)
df_predicciones.to_excel(os.path.join(PATH_OUTPUT_DIR, 'augmented_predicciones.xlsx'), index=False)
print(f"  Guardado: augmented_predicciones.xlsx")

resumen_metricas = df_comparativa.groupby(['fuente', 'target']).agg({
    'MAE': ['mean', 'min', 'max'],
    'R2': ['mean', 'min', 'max'],
    'MAPE': ['mean', 'min', 'max']
}).reset_index()
resumen_metricas.columns = ['fuente', 'target', 'MAE_mean', 'MAE_min', 'MAE_max',
                            'R2_mean', 'R2_min', 'R2_max',
                            'MAPE_mean', 'MAPE_min', 'MAPE_max']
resumen_metricas.to_excel(os.path.join(PATH_OUTPUT_DIR, 'augmented_metricas_resumen.xlsx'), index=False)
print(f"  Guardado: augmented_metricas_resumen.xlsx")

print(f"\nArchivos exportados en: {PATH_OUTPUT_DIR}")
print("Pipeline de modelos con augmentation completado.")

  Guardado: augmented_comparativa_modelos.xlsx
  Guardado: augmented_predicciones.xlsx
  Guardado: augmented_metricas_resumen.xlsx

Archivos exportados en: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\resultados
Pipeline de modelos con augmentation completado.
